In [22]:
import pandas as pd

In [39]:
df= pd.read_excel("backend/data/admin_units.xlsx")

In [40]:
df

,"МОНГОЛ УЛСЫН ЗАСАГ ЗАХИРГАА, НУТАГ ДЭВСГЭРИЙН НЭГЖ, бүс, аймаг, нийслэл, жилээр",Unnamed: 1,Unnamed: 2,Unnamed: 3
0,Үзүүлэлт,Бүс,Код,"2,025"
1,"Сум, дүүргийн тоо",Улсын дүн,0,339
2,"Сум, дүүргийн тоо",Аймгийн дүн,6,330
3,"Сум, дүүргийн тоо",Баруун бүс,1,91
4,"Сум, дүүргийн тоо",Баян-Өлгий,183,13
...,...,...,...,...
112,Хүн амын нягтрал(1 км2-т ногдох хүн),Дорнод,421,1
113,Хүн амын нягтрал(1 км2-т ногдох хүн),Сүхбаатар,422,1
114,Хүн амын нягтрал(1 км2-т ногдох хүн),Хэнтий,423,1
115,Хүн амын нягтрал(1 км2-т ногдох хүн),Улаанбаатар,5,382


In [41]:
df.columns

Index(['МОНГОЛ УЛСЫН ЗАСАГ ЗАХИРГАА, НУТАГ ДЭВСГЭРИЙН НЭГЖ, бүс, аймаг, нийслэл, жилээр',
       'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3'],
      dtype='object')

In [42]:
pd.options.display.float_format = '{:,.0f}'.format

In [43]:


# 1. Багануудаа шинэчлэн нэрлэх
df = df.rename(columns={
    'МОНГОЛ УЛСЫН ЗАСАГ ЗАХИРГАА, НУТАГ ДЭВСГЭРИЙН НЭГЖ, бүс, аймаг, нийслэл, жилээр': 'Аймаг, нийслэл',
    'Unnamed: 1': 'Бүс',
    'Unnamed: 2': 'Код',
    'Unnamed: 3': '2025 он'
})

# 2. Илүүдэл мөрийг устгах, цэвэрлэх
df = df.iloc[1:].reset_index(drop=True)
df['Бүс'] = df['Бүс'].str.strip()  # Сул зайг цэвэрлэх

# 3. Бүсээр мэдээлэл авдаг функц (DataFrame буцаадаг)
def busiin_medeelel_avah(busiin_ner):
    # Хайлт хийх
    result = df[df['Бүс'].str.contains(busiin_ner, na=False, case=False)]
    
    if result.empty:
        # Хэрэв юу ч олдоогүй бол хоосон DataFrame буцаана
        print(f"⚠️ '{busiin_ner}' нэртэй бүс олдсонгүй.")
        return pd.DataFrame() 
    
    return result

# --- Ашиглах хэсэг ---
ner = input("Асуух бүсийн нэрээ оруулна уу: ")
search_result = busiin_medeelel_avah(ner)

# Хэрэв үр дүн хоосон биш бол хүснэгтээр харуулна
if not search_result.empty:
    print(f"\n✅ {ner} бүсийн мэдээлэл:")
    # Хэрэв та Jupyter дээр байгаа бол: display(search_result)
    print(search_result.to_string(index=False)) # Индексгүйгээр цэвэрхэн хэвлэх

Асуух бүсийн нэрээ оруулна уу:  увс



✅ увс бүсийн мэдээлэл:
                      Аймаг, нийслэл Бүс Код  2025 он
                   Сум, дүүргийн тоо Увс 185       19
                    Баг, хорооны тоо Увс 185       93
   Нутаг дэвсгэрийн хэмжээ (мян.км2) Увс 185       70
Хүн амын нягтрал(1 км2-т ногдох хүн) Увс 185        1


In [45]:
import pandas as pd
import streamlit as st

def load_data():
    df = pd.read_excel("backend/data/admin_units.xlsx")
    
    # Багануудыг шинэчлэн нэрлэх
    df = df.rename(columns={
        'МОНГОЛ УЛСЫН ЗАСАГ ЗАХИРГАА, НУТАГ ДЭВСГЭРИЙН НЭГЖ, бүс, аймаг, нийслэл, жилээр': 'Аймаг_нийслэл',
        'Unnamed: 1': 'Бүс',
        'Unnamed: 2': 'Код',
        'Unnamed: 3': '2025 он'
    })
    
    # Илүүдэл мөр устгах ба цэвэрлэгээ
    df = df.iloc[1:].reset_index(drop=True)
    df['Бүс'] = df['Бүс'].str.strip()
    df['Аймаг_нийслэл'] = df['Аймаг_нийслэл'].str.strip()
    
    # '2025 он' баганыг тоо руу хөрвүүлэх (Нийлбэр гаргахад бэлдэх)
    df['2025 он'] = pd.to_numeric(df['2025 он'], errors='coerce').fillna(0)
    
    return df

df = load_data()

# --- 2. ФУНКЦ ТОДОРХОЙЛОХ ХЭСЭГ ---
def hailt_hiiх(search_query):
    """Бүс эсвэл Аймаг_нийслэл баганаас хайлт хийх"""
    # Хоёр баганаас зэрэг хайлт хийх (OR логик)
    mask = (
        df['Бүс'].str.contains(search_query, na=False, case=False) | 
        df['Аймаг_нийслэл'].str.contains(search_query, na=False, case=False)
    )
    return df[mask]

# --- 3. STREAMLIT UI ---
st.set_page_config(page_title="Статистик хайлт", layout="wide")
st.title("🗺️ Бүс болон Аймгийн мэдээлэл хайх")

# Хайлтын талбар
ner = st.text_input("Хайх утгаа оруулна уу (Жишээ нь: Баруун бүс, Архангай):")

if ner:
    search_result = hailt_hiiх(ner)
    
    if not search_result.empty:
        st.success(f"✅ '{ner}' нэртэй {len(search_result)} илэрц олдлоо:")
        
        # 1. Хүснэгт харуулах
        st.dataframe(search_result, use_container_width=True)
        
        # 2. 2025 оны нийлбэр дүнг харуулах (Metric)
        st.divider()
        col1, col2 = st.columns(2)
        
        niilber_2025 = search_result['2025 он'].sum()
        
        with col1:
            st.metric(label="🔎 Шүүгдсэн илэрцийн нийт дүн (2025)", value=f"{niilber_2025:,.1f}")
        
        with col2:
            # Сонгосон хэсгийн нийт дата дахь эзлэх хувийг харуулж болно
            total_all = df['2025 он'].sum()
            ezleh_huvi = (niilber_2025 / total_all) * 100
            st.metric(label="Нийт дүнд эзлэх хувь", value=f"{ezleh_huvi:.2f}%")
            
    else:
        st.warning(f"⚠️ '{ner}' нэртэй мэдээлэл олдсонгүй.")
else:
    st.info("Дээрх талбарт хайх утгаа бичнэ үү.")
    st.write("Нийт мэдээллийн сан:", df)

2026-04-27 21:04:40.629 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-27 21:04:40.639 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-27 21:04:40.641 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-27 21:04:40.653 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-27 21:04:40.656 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-27 21:04:40.657 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-27 21:04:40.658 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-27 21:04:40.660 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [ ]:
import pandas as pd
import streamlit as st

# --- 1. ӨГӨГДӨЛ УНШИХ БА ЦЭВЭРЛЭХ (Таны үндсэн код) ---
@st.cache_data
def load_data():
    # Файлын замыг өөрийнхөөрөө засаарай
    df_raw = pd.read_excel("data/admin_units.xlsx") 
    
    df = df_raw.rename(columns={
        'МОНГОЛ УЛСЫН ЗАСАГ ЗАХИРГАА, НУТАГ ДЭВСГЭРИЙН НЭГЖ, бүс, аймаг, нийслэл, жилээр': 'Аймаг_нийслэл',
        'Unnamed: 1': 'Бүс',
        'Unnamed: 2': 'Код',
        'Unnamed: 3': '2025 он'
    })
    
    df = df.iloc[1:].reset_index(drop=True)
    df['Бүс'] = df['Бүс'].str.strip()
    # Нэмэлтээр Аймаг_нийслэл баганыг цэвэрлэх
    df['Аймаг_нийслэл'] = df['Аймаг_нийслэл'].str.strip()
    
    # 2025 он баганыг тоо руу хөрвүүлэх (Нийлбэр гаргахад зайлшгүй шаардлагатай)
    df['2025 он'] = pd.to_numeric(df['2025 он'], errors='coerce').fillna(0)
    
    return df

df = load_data()

# --- 2. ӨРГӨТГӨСӨН ХАЙЛТЫН ФУНКЦ (Үндсэн логикийг хадгалсан) ---
def busiin_medeelel_avah(hailt_utga):
    # 'Бүс' ЭСВЭЛ 'Аймаг_нийслэл' баганаас зэрэг хайна
    mask = (
        df['Бүс'].str.contains(hailt_utga, na=False, case=False) | 
        df['Аймаг_нийслэл'].str.contains(hailt_utga, na=False, case=False)
    )
    result = df[mask]
    
    if result.empty:
        return pd.DataFrame()
    return result

# --- 3. STREAMLIT UI ХЭСЭГ ---
st.title("📊 Статистик мэдээллийн нэгдсэн хайлт")

ner = st.text_input("Хайх бүс эсвэл аймгийн нэрээ оруулна уу:", placeholder="Жишээ нь: Архангай эсвэл Баруун бүс")

if ner:
    search_result = busiin_medeelel_avah(ner)
    
    if not search_result.empty:
        st.success(f"✅ '{ner}' нэрээр хайсан үр дүн:")
        
        # Үр дүнг хүснэгтээр харуулах
        st.dataframe(search_result, use_container_width=True)
        
        # --- НЭМЭЛТ: НИЙЛБЭР ДҮНГ ХАРУУЛАХ ---
        total_2025 = search_result['2025 он'].sum()
        
        st.markdown("---")
        col1, col2 = st.columns(2)
        with col1:
            st.metric(label=f"'{ner}' - 2025 оны нийт дүн", value=f"{total_2025:,.1f}")
        with col2:
            st.info(f"Нийт {len(search_result)} мөр мэдээлэл олдлоо.")
            
    else:
        st.warning(f"⚠️ '{ner}' нэртэй мэдээлэл олдсонгүй.")
else:
    st.info("Хайх утгаа оруулна уу.")